# Delta Lake Fundamentals Lab

## Overview
This project is a hands-on introduction to core Delta Lake concepts in Databricks.

The goal is to understand how Delta Lake behaves more like a database than a plain data lake by supporting versioning, updates, deletes, and time travel.

## What this notebook covers
- Creating a Delta table
- Inserting data
- Updating and deleting rows
- Viewing Delta table history
- Querying previous versions with time travel
- Using `MERGE` for upserts
- Demonstrating schema evolution
- Understanding managed vs external table concepts

## Why this matters
Delta Lake is a foundational concept in Databricks data engineering workflows.  
This notebook is designed as a beginner-friendly training lab and reference project.

## Data used
This project uses small custom demo tables created directly in the notebook so the behavior is easy to follow.

In [0]:
DROP TABLE IF EXISTS demo_delta;
DROP TABLE IF EXISTS demo_delta_source;

-- This makes the notebook rerunnable.
-- If you run it again tomorrow, it won’t break because old tables already exist.

## Step 1: Create a Delta table

In this step, we create a simple Delta table called `demo_delta`.

This table will be used to demonstrate how Delta Lake handles inserts, updates, deletes, version history, and schema changes.

In [0]:
CREATE OR REPLACE TABLE demo_delta (
    id INT,
    name STRING
)
USING DELTA;


### What this does

Creates a **managed Delta table**.

### Why this matters

- `USING DELTA` tells Databricks to store the table in Delta format  
- Because no `LOCATION` was specified, this is a **managed table**  
- Databricks controls where the data is physically stored  

### Managed vs External (quick note)

- **Managed table** → Databricks manages storage  
- **External table** → user defines storage location (e.g. `s3://...`)

## Step 2: Insert the first row

We insert one row into the Delta table.

This creates the first data version after table creation.

In [0]:
INSERT INTO demo_delta VALUES (1, 'Bob');

num_affected_rows,num_inserted_rows
1,1


### What this does

Adds one record.

This should create:

a new version in Delta history
underlying Delta data/log changes behind the scenes

## Step 3: Query the table

Now we check the current contents of the table.

In [0]:
SELECT * FROM demo_delta;

id,name
1,Bob


## Step 4: Update data

Delta Lake does not overwrite the original file directly.

Instead, Delta tracks changes as new versions and records the update in the transaction log.

In [0]:
-- Update the current version of the row
UPDATE demo_delta
SET name = 'Robert'
WHERE id = 1;

num_affected_rows
1


## Step 5: View Delta history

`DESCRIBE HISTORY` shows the version history of the table.

This is one of the key features that makes Delta Lake feel more like a database than a plain file-based data lake.

In [0]:
DESCRIBE HISTORY demo_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-21T23:41:06.000Z,78446220094812,jenniferarias414@gmail.com,UPDATE,"Map(predicate -> [""(id#13495 = 1)""])",null,List(3954992076238518),a8d72095-e29a-4dfc-be38-99facdab2d6b,0421-231747-k7ifw00w-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 846, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2836, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1176, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1591, rewriteTimeMs -> 1660)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-21T23:40:59.000Z,78446220094812,jenniferarias414@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3954992076238518),b554deb8-9cdb-4e2f-9581-ff6243ad8ad0,0421-231747-k7ifw00w-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 846)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-21T23:40:57.000Z,78446220094812,jenniferarias414@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ad9d20c9-1aae-4081-8f78-cd36a85047d6"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-f9c40bdd-2d6f-4a61-87e1-2be530342e07""}, statsOnLoad -> false)",null,List(3954992076238518),f8b6912a-69f9-4899-8dee-ad41485b1a64,0421-231747-k7ifw00w-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


### What to Notice

Each operation creates a new **version** of the table.

You should see operations like:
- CREATE OR REPLACE TABLE  
- WRITE  
- UPDATE  

And each one should have a version number.

Each version represents a snapshot of the table at a point in time.

## Step 6: Time travel

Delta Lake allows us to query older versions of the table.

This is called time travel.

In this example, version 1 should show the data before the update.

In [0]:
-- Query an older version of the table
SELECT * FROM demo_delta VERSION AS OF 1;

id,name
1,Bob


### What to notice

This should show:
Bob

even though the current version says:
Robert

That proves Delta kept the earlier version.

## Step 7: Delete a row

Delta Lake also supports delete operations.

Again, the table is versioned, so this change is tracked rather than simply destroying history.

In [0]:
DELETE FROM demo_delta
WHERE id = 1;

num_affected_rows
1


In [0]:
SELECT * FROM demo_delta;

id,name


### What to notice

The table should now be empty in the current version.

## Step 8: Insert fresh rows for merge demo

Now we add a few rows back into the table so we can demonstrate `MERGE`, which is commonly used for upserts in data engineering workflows.

In [0]:
INSERT INTO demo_delta VALUES
(1, 'Bob'),
(2, 'Jenny');

num_affected_rows,num_inserted_rows
2,2


## Step 9: Create a source table for MERGE

A source table is often used in real ETL workflows to represent incoming records that may update existing rows or add new ones.

In [0]:
CREATE OR REPLACE TABLE demo_delta_source (
    id INT,
    name STRING
)
USING DELTA;

In [0]:
INSERT INTO demo_delta_source VALUES
(2, 'Jenny Updated'),
(3, 'Charlie');

num_affected_rows,num_inserted_rows
2,2


## Step 10: MERGE (upsert behavior)

`MERGE` allows us to:
- update matching rows
- insert non-matching rows

This is one of the most important Delta Lake features for real-world pipelines.

In [0]:
MERGE INTO demo_delta AS target
USING demo_delta_source AS source
ON target.id = source.id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name
WHEN NOT MATCHED THEN
  INSERT (id, name)
  VALUES (source.id, source.name);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
SELECT * FROM demo_delta ORDER BY id;

id,name
1,Bob
2,Jenny Updated
3,Charlie


### What to Notice

You should now see:
1, Bob
2, Jenny Updated
3, Charlie

That means:

existing row was updated
new row was inserted

## Step 11: Schema evolution

Schema evolution means the structure of the table can change over time.

Here, we add a new column to demonstrate that Delta tables can evolve.

In [0]:
ALTER TABLE demo_delta ADD COLUMNS (status STRING);

In [0]:
UPDATE demo_delta
SET status = 'active'
WHERE id IN (1, 2, 3);

num_affected_rows
3


In [0]:
SELECT * FROM demo_delta ORDER BY id;

id,name,status
1,Bob,active
2,Jenny Updated,active
3,Charlie,active


### What to Notice

The table now has a new column:
status

This is a simple schema evolution example.

## Step 12: Final history check

We review the history one more time to see how Delta tracked all of the operations in this notebook.

In [0]:
DESCRIBE HISTORY demo_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-21T23:41:42.000Z,78446220094812,jenniferarias414@gmail.com,UPDATE,"Map(predicate -> [""id#14892 IN (1,2,3)""])",null,List(3954992076238518),3dc2aabb-9fa0-4ce5-81bb-a8fd09d164cf,0421-231747-k7ifw00w-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1733, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4707, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1297, numAddedFiles -> 1, numUpdatedRows -> 3, numAddedBytes -> 1887, rewriteTimeMs -> 3410)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-21T23:41:35.000Z,78446220094812,jenniferarias414@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""status"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",null,List(3954992076238518),3fcfe503-c2e3-44a0-881c-447fe036b1fb,0421-231747-k7ifw00w-v2n,6,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-21T23:41:33.000Z,78446220094812,jenniferarias414@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3954992076238518),50dcbf6a-bf7d-4ed4-a960-57d0c86ac265,0421-231747-k7ifw00w-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2503, p25FileSize -> 1733, numDeletionVectorsRemoved -> 1, minFileSize -> 1733, numAddedFiles -> 1, maxFileSize -> 1733, p75FileSize -> 1733, p50FileSize -> 1733, numAddedBytes -> 1733)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-21T23:41:30.000Z,78446220094812,jenniferarias414@gmail.com,MERGE,"Map(predicate -> [""(id#14412 = id#14414)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3954992076238518),50dcbf6a-bf7d-4ed4-a960-57d0c86ac265,0421-231747-k7ifw00w-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1639, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4064, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1606, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2408)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-21T23:41:19.000Z,78446220094812,jenniferarias414@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3954992076238518),12db7413-60bd-4e11-bd7d-4a1c89765eaf,0421-231747-k7ifw00w-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 864)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-21T23:41:15.000Z,78446220094812,jenniferarias414@gmail.com,DELETE,"Map(predicate -> [""(id#14010 = 1)""])",null,List(3954992076238518),bd0d8f13-eeb6-4284-bd97-7291b2f69d5a,0421-231747-k7ifw00w-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1591, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1721, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1552, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 168)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-21T23:41:06.000Z,78446220094812,jenniferarias414@gmail.com,UPDATE,"Map(predicate -> [""(id#13495 = 1)""])",null,List(3954

## Final Takeaways

This project demonstrates that Delta Lake:

- tracks table changes using version history  
- supports safe updates and deletes without overwriting data  
- enables time travel to previous table states  
- supports merge-based upsert workflows  
- allows schema changes over time  

### In simple terms

Delta Lake is a table format and transaction layer that makes data lakes behave more like databases.

### Why this matters

These capabilities are commonly used in real-world data engineering workflows such as:

- ETL pipelines  
- CDC / upsert patterns  
- audit and history tracking  
- building reliable analytics tables  